# Synthetic SBM — Module-Compatible Verification Notebook

This notebook reproduces the original synthetic SBM experiment while only replacing the parts that are considered safe to modularize with `community_detection_utils.py`.

The goal is to check whether using the shared module preserves the original ARI/NMI results.

In [ ]:
# ============================================================
# CELL 1 — INSTALL, IMPORTS, AND MODULE
# ============================================================

!pip install -q \
    networkx \
    python-louvain \
    scikit-learn \
    pandas \
    numpy \
    GraphRicciCurvature \
    python-igraph \
    leidenalg

import time
import warnings

import numpy as np
import pandas as pd
import networkx as nx
import community as community_louvain

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

from GraphRicciCurvature.OllivierRicci import OllivierRicci

%load_ext autoreload
%autoreload 2

import community_detection_utils as cdu

warnings.filterwarnings("ignore")

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("notebook_output_data/synthetic_module_verification")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# CELL 2 — FIXED EXPERIMENTAL PARAMETERS
# ============================================================

FIXED_PARAMS = dict(
    n_communities=4,
    community_size=90,
    p_in=0.15,
    p_out=0.022,
    bridge_blocks=10,
    bridge_block_size=8,
)

PCT_LRC_ONLY = 50
PCT_LRC_COMBO = 15
PCT_ORC = 20

ALPHA_ORC = 0.5

RICCI_FLOW_STEPS = 3
RICCI_FLOW_ETA = 1.0

N_SEEDS = 10
RANDOM_SEED = 42

In [ ]:
# ============================================================
# CELL 3 — SYNTHETIC GRAPH GENERATION
# ============================================================

def generate_graph(seed, **params):

    rng = np.random.default_rng(seed)

    sizes = (
        [params["community_size"]]
        * params["n_communities"]
    )

    probs = np.full(
        (
            params["n_communities"],
            params["n_communities"],
        ),
        params["p_out"],
    )

    np.fill_diagonal(
        probs,
        params["p_in"],
    )

    G = nx.stochastic_block_model(
        sizes,
        probs,
        seed=seed,
    )

    labels = {}
    community_nodes = []

    start = 0

    for c, size in enumerate(sizes):

        nodes = list(
            range(
                start,
                start + size,
            )
        )

        community_nodes.append(nodes)

        for node in nodes:
            labels[node] = c

        start += size

    # Triangle-rich inter-community bridge blocks
    for _ in range(params["bridge_blocks"]):

        c1, c2 = rng.choice(
            params["n_communities"],
            size=2,
            replace=False,
        )

        block1 = rng.choice(
            community_nodes[c1],
            params["bridge_block_size"],
            replace=False,
        )

        block2 = rng.choice(
            community_nodes[c2],
            params["bridge_block_size"],
            replace=False,
        )

        for u in block1:
            for v in block2:
                G.add_edge(int(u), int(v))

    # MODULE FUNCTION #1
    G = cdu.get_gcc(G)

    labels = {
        node: labels[node]
        for node in G.nodes()
    }

    # MODULE FUNCTION #2
    G = cdu.ensure_edge_weights(
        G,
        weight="weight",
        default_weight=1.0,
        copy_graph=False,
    )

    return G, labels

In [ ]:
# ============================================================
# CELL 4 — LRC USING SHARED MODULE
# ============================================================

def lrc_prune(G, pct):

    H = G.copy()

    # Compute LRC using the shared implementation
    cdu.compute_LRC(
        H,
        attribute="LRC",
    )

    # Prune low-LRC edges using the shared implementation
    H_pruned, cutoff, removed = cdu.prune_by_percentile(
        H,
        attr="LRC",
        pct=pct,
    )

    return H_pruned, cutoff, removed

In [ ]:
# ============================================================
# CELL 5 — ORC + RICCI FLOW USING SHARED MODULE
# ============================================================

def orc_flow_prune(
    G,
    pct=20,
    alpha=0.5,
    flow_steps=3,
    eta=1.0,   # kept only for compatibility with existing calls
):

    # Run Ricci flow using shared module
    H_flow = cdu.run_orc_flow_components(
        G,
        alpha=alpha,
        iterations=flow_steps,
        method="Sinkhorn",
        weight="weight",
    )

    # Prune low final ORC edges using shared module
    H_pruned, cutoff, removed = cdu.prune_by_percentile(
        H_flow,
        attr="ricciCurvature",
        pct=pct,
    )

    # Keep all nodes and connected components after ORC pruning.
    # Components separated by pruning may represent meaningful communities.
    return H_pruned, cutoff, removed

In [ ]:
# ============================================================
# CELL 6 — EVALUATION
# ============================================================

def evaluate_graph(G, labels):

    # MODULE FUNCTION #4
    partition = cdu.run_louvain(
        G,
        weight="weight",
        seed=RANDOM_SEED,
    )

    nodes = sorted(G.nodes())

    y_true = [
        labels[node]
        for node in nodes
    ]

    y_pred = [
        partition[node]
        for node in nodes
    ]

    ari = adjusted_rand_score(
        y_true,
        y_pred,
    )

    nmi = normalized_mutual_info_score(
        y_true,
        y_pred,
    )

    modularity = community_louvain.modularity(
        partition,
        G,
        weight="weight",
    )

    return (
        ari,
        nmi,
        modularity,
    )

In [ ]:
# ============================================================
# CELL 7 — GRAPH STATISTICS
# ============================================================

def graph_stats(G):

    # MODULE FUNCTION #5
    stats = cdu.basic_graph_stats(
        G,
        include_path_metrics=False,
    )

    return (
        stats["nodes"],
        stats["edges"],
        stats["density"],
        stats["avg_degree"],
    )

In [ ]:
# ============================================================
# CELL 8 — THREE PIPELINES
# ============================================================

def run_lrc_only(
    G,
    labels,
    pct,
):

    t0 = time.time()

    t1 = time.time()

    H, _, removed = lrc_prune(
        G,
        pct,
    )

    lrc_time = time.time() - t1

    t2 = time.time()

    ari, nmi, mod = evaluate_graph(
        H,
        labels,
    )

    louvain_time = time.time() - t2

    return {
        "pipeline": "LRC-only",
        "ARI": ari,
        "NMI": nmi,
        "Modularity": mod,
        "LRC_time": lrc_time,
        "ORC_flow_time": 0.0,
        "Louvain_time": louvain_time,
        "Total_time": time.time() - t0,
        "removed_lrc": removed,
        "removed_orc": 0,
        "V": H.number_of_nodes(),
        "E": H.number_of_edges(),
    }


def run_orc_only(
    G,
    labels,
    pct,
    alpha,
    flow_steps,
    eta,
):

    t0 = time.time()

    t1 = time.time()

    H, _, removed = orc_flow_prune(
        G,
        pct,
        alpha,
        flow_steps,
        eta,
    )

    orc_flow_time = time.time() - t1

    t2 = time.time()

    ari, nmi, mod = evaluate_graph(
        H,
        labels,
    )

    louvain_time = time.time() - t2

    return {
        "pipeline": "ORC-only",
        "ARI": ari,
        "NMI": nmi,
        "Modularity": mod,
        "LRC_time": 0.0,
        "ORC_flow_time": orc_flow_time,
        "Louvain_time": louvain_time,
        "Total_time": time.time() - t0,
        "removed_lrc": 0,
        "removed_orc": removed,
        "V": H.number_of_nodes(),
        "E": H.number_of_edges(),
    }


def run_combo(
    G,
    labels,
    pct_lrc,
    pct_orc,
    alpha,
    flow_steps,
    eta,
):

    t0 = time.time()

    t1 = time.time()

    H1, _, removed_l = lrc_prune(
        G,
        pct_lrc,
    )

    lrc_time = time.time() - t1

    t2 = time.time()

    H2, _, removed_o = orc_flow_prune(
        H1,
        pct_orc,
        alpha,
        flow_steps,
        eta,
    )

    orc_flow_time = time.time() - t2
    sparsification_time = time.time() - t0

    t3 = time.time()

    ari, nmi, mod = evaluate_graph(
        H2,
        labels,
    )

    louvain_time = time.time() - t3

    return {
        "pipeline": "Combo (LRC->ORC)",
        "ARI": ari,
        "NMI": nmi,
        "Modularity": mod,
        "LRC_time": lrc_time,
        "ORC_flow_time": orc_flow_time,
        "Louvain_time": louvain_time,
        "Sparsification_time": sparsification_time,
        "Total_time": sparsification_time,
        "removed_lrc": removed_l,
        "removed_orc": removed_o,
        "V": H2.number_of_nodes(),
        "E": H2.number_of_edges(),
    }

In [ ]:
# ============================================================
# CELL 9 — MAIN EXPERIMENT
# ============================================================

all_rows = []

print(
    f"Running {N_SEEDS} independent graph realizations...\n"
)

print(
    f"Ricci flow steps T = {RICCI_FLOW_STEPS}, "
    f"eta = {RICCI_FLOW_ETA}\n"
)

for seed in range(
    1,
    N_SEEDS + 1,
):

    G, labels = generate_graph(
        seed,
        **FIXED_PARAMS,
    )

    r_lrc = run_lrc_only(
        G,
        labels,
        pct=PCT_LRC_ONLY,
    )

    r_orc = run_orc_only(
        G,
        labels,
        pct=PCT_ORC,
        alpha=ALPHA_ORC,
        flow_steps=RICCI_FLOW_STEPS,
        eta=RICCI_FLOW_ETA,
    )

    r_combo = run_combo(
        G,
        labels,
        pct_lrc=PCT_LRC_COMBO,
        pct_orc=PCT_ORC,
        alpha=ALPHA_ORC,
        flow_steps=RICCI_FLOW_STEPS,
        eta=RICCI_FLOW_ETA,
    )

    for result in [
        r_lrc,
        r_orc,
        r_combo,
    ]:
        result["seed"] = seed

    all_rows.extend(
        [
            r_lrc,
            r_orc,
            r_combo,
        ]
    )

    print(
        f"Seed {seed:2d} | "
        f"LRC ARI={r_lrc['ARI']:.3f} | "
        f"ORC ARI={r_orc['ARI']:.3f} | "
        f"Combo ARI={r_combo['ARI']:.3f} | "
        f"Combo time={r_combo['Total_time']:.2f}s | "
        f"ORC time={r_orc['Total_time']:.2f}s"
    )

df = pd.DataFrame(
    all_rows
)

In [ ]:
# ============================================================
# CELL 10 — SUMMARY
# ============================================================

summary_cols = [
    "ARI",
    "NMI",
    "Modularity",
    "Total_time",
]

summary = (
    df
    .groupby("pipeline")[summary_cols]
    .agg(["mean", "std"])
    .round(4)
)

print("\n" + "=" * 70)
print("RESULTS (mean +/- std across seeds)")
print("=" * 70)

display(summary)

rows_fmt = []

for pipe in [
    "LRC-only",
    "ORC-only",
    "Combo (LRC->ORC)",
]:

    sub = df[
        df["pipeline"] == pipe
    ]

    rows_fmt.append(
        {
            "Pipeline": pipe,

            "ARI mean+/-std":
                cdu.mean_pm_std(
                    sub["ARI"],
                    digits=3,
                ),

            "NMI mean+/-std":
                cdu.mean_pm_std(
                    sub["NMI"],
                    digits=3,
                ),

            "Time mean+/-std":
                cdu.mean_pm_std(
                    sub["Total_time"],
                    digits=2,
                ),
        }
    )

df_fmt = pd.DataFrame(
    rows_fmt
).set_index("Pipeline")

display(df_fmt)

In [ ]:
# ============================================================
# CELL 11 — VERIFY AGAINST ORIGINAL RESULTS
# ============================================================

EXPECTED = {
    "LRC-only": {
        "ARI": 0.6320,
        "NMI": 0.6371,
    },

    "ORC-only": {
        "ARI": 0.8544,
        "NMI": 0.8357,
    },

    "Combo (LRC->ORC)": {
        "ARI": 0.8049,
        "NMI": 0.7846,
    },
}

print("=" * 75)
print("COMPARISON WITH ORIGINAL SYNTHETIC RESULTS")
print("=" * 75)

for pipeline, expected in EXPECTED.items():

    subset = df[
        df["pipeline"] == pipeline
    ]

    ari_new = subset["ARI"].mean()
    nmi_new = subset["NMI"].mean()

    print(f"\n{pipeline}")

    print(
        f"  Original ARI : "
        f"{expected['ARI']:.4f}"
    )

    print(
        f"  New ARI      : "
        f"{ari_new:.4f}"
    )

    print(
        f"  ARI difference: "
        f"{ari_new - expected['ARI']:+.6f}"
    )

    print(
        f"  Original NMI : "
        f"{expected['NMI']:.4f}"
    )

    print(
        f"  New NMI      : "
        f"{nmi_new:.4f}"
    )

    print(
        f"  NMI difference: "
        f"{nmi_new - expected['NMI']:+.6f}"
    )

In [ ]:
# ============================================================
# CELL 12 — PARAMETER SENSITIVITY ANALYSIS
# ============================================================

import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Sensitivity settings
# ------------------------------------------------------------

# Same independent graph realizations used for each parameter value
SENSITIVITY_SEEDS = [1, 2, 3, 4, 5]

# Parameter values to investigate
LRC_PRUNING_VALUES = [0, 5, 10, 15, 20, 25, 30]

ORC_PRUNING_VALUES = [5, 10, 15, 20, 25, 30]

ALPHA_VALUES = [0.00, 0.25, 0.50, 0.75]


# Main/default configuration
DEFAULT_LRC_PRUNING = PCT_LRC_COMBO
DEFAULT_ORC_PRUNING = PCT_ORC
DEFAULT_ALPHA = ALPHA_ORC


print("Default configuration:")
print(f"  LRC pruning = {DEFAULT_LRC_PRUNING}%")
print(f"  ORC pruning = {DEFAULT_ORC_PRUNING}%")
print(f"  alpha       = {DEFAULT_ALPHA}")
print(f"  flow steps  = {RICCI_FLOW_STEPS}")
print(f"  eta         = {RICCI_FLOW_ETA}")
print(f"  seeds       = {SENSITIVITY_SEEDS}")


# ============================================================
# 2. Evaluate one Combo configuration
# ============================================================

def evaluate_combo_configuration(
    lrc_pct,
    orc_pct,
    alpha,
    seeds,
):
    """
    Run one LRC->ORC parameter configuration across
    several independent synthetic SBM realizations.
    """

    rows = []

    for seed in seeds:

        # Same graph-generation procedure as the main experiment
        G, labels = generate_graph(
            seed,
            **FIXED_PARAMS,
        )

        result = run_combo(
            G=G,
            labels=labels,
            pct_lrc=lrc_pct,
            pct_orc=orc_pct,
            alpha=alpha,
            flow_steps=RICCI_FLOW_STEPS,
            eta=RICCI_FLOW_ETA,
        )

        rows.append(
            {
                "seed": seed,
                "lrc_pct": lrc_pct,
                "orc_pct": orc_pct,
                "alpha": alpha,

                "ARI": result["ARI"],
                "NMI": result["NMI"],
                "Runtime": result["Total_time"],

                "V": result["V"],
                "E": result["E"],

                "removed_lrc":
                    result["removed_lrc"],

                "removed_orc":
                    result["removed_orc"],
            }
        )

    return rows


# ============================================================
# 3. LRC pruning sensitivity
# ============================================================

lrc_sweep_rows = []

print("\n" + "=" * 70)
print("LRC PRUNING SENSITIVITY")
print("=" * 70)

for lrc_pct in LRC_PRUNING_VALUES:

    print(
        f"LRC={lrc_pct:>2}% | "
        f"ORC={DEFAULT_ORC_PRUNING}% | "
        f"alpha={DEFAULT_ALPHA}"
    )

    rows = evaluate_combo_configuration(
        lrc_pct=lrc_pct,
        orc_pct=DEFAULT_ORC_PRUNING,
        alpha=DEFAULT_ALPHA,
        seeds=SENSITIVITY_SEEDS,
    )

    lrc_sweep_rows.extend(
        rows
    )


df_lrc_sensitivity = pd.DataFrame(
    lrc_sweep_rows
)


# ============================================================
# 4. ORC pruning sensitivity
# ============================================================

orc_sweep_rows = []

print("\n" + "=" * 70)
print("ORC PRUNING SENSITIVITY")
print("=" * 70)

for orc_pct in ORC_PRUNING_VALUES:

    print(
        f"LRC={DEFAULT_LRC_PRUNING}% | "
        f"ORC={orc_pct:>2}% | "
        f"alpha={DEFAULT_ALPHA}"
    )

    rows = evaluate_combo_configuration(
        lrc_pct=DEFAULT_LRC_PRUNING,
        orc_pct=orc_pct,
        alpha=DEFAULT_ALPHA,
        seeds=SENSITIVITY_SEEDS,
    )

    orc_sweep_rows.extend(
        rows
    )


df_orc_sensitivity = pd.DataFrame(
    orc_sweep_rows
)


# ============================================================
# 5. Alpha sensitivity
# ============================================================

alpha_sweep_rows = []

print("\n" + "=" * 70)
print("ALPHA SENSITIVITY")
print("=" * 70)

for alpha in ALPHA_VALUES:

    print(
        f"LRC={DEFAULT_LRC_PRUNING}% | "
        f"ORC={DEFAULT_ORC_PRUNING}% | "
        f"alpha={alpha:.2f}"
    )

    rows = evaluate_combo_configuration(
        lrc_pct=DEFAULT_LRC_PRUNING,
        orc_pct=DEFAULT_ORC_PRUNING,
        alpha=alpha,
        seeds=SENSITIVITY_SEEDS,
    )

    alpha_sweep_rows.extend(
        rows
    )


df_alpha_sensitivity = pd.DataFrame(
    alpha_sweep_rows
)


# ============================================================
# 6. Summaries: mean +/- standard deviation
# ============================================================

def summarize_sensitivity(
    dataframe,
    parameter_column,
):

    return (
        dataframe
        .groupby(
            parameter_column,
            as_index=False,
        )
        .agg(
            ARI_mean=("ARI", "mean"),
            ARI_std=("ARI", "std"),

            NMI_mean=("NMI", "mean"),
            NMI_std=("NMI", "std"),

            Runtime_mean=("Runtime", "mean"),
            Runtime_std=("Runtime", "std"),

            V_mean=("V", "mean"),
            V_std=("V", "std"),

            E_mean=("E", "mean"),
            E_std=("E", "std"),

            Removed_LRC_mean=(
                "removed_lrc",
                "mean",
            ),

            Removed_ORC_mean=(
                "removed_orc",
                "mean",
            ),
        )
        .sort_values(
            parameter_column
        )
        .reset_index(
            drop=True
        )
    )


lrc_sensitivity_summary = (
    summarize_sensitivity(
        df_lrc_sensitivity,
        "lrc_pct",
    )
)

orc_sensitivity_summary = (
    summarize_sensitivity(
        df_orc_sensitivity,
        "orc_pct",
    )
)

alpha_sensitivity_summary = (
    summarize_sensitivity(
        df_alpha_sensitivity,
        "alpha",
    )
)


print(
    "\nLRC pruning sensitivity:"
)
display(
    lrc_sensitivity_summary
)

print(
    "\nORC pruning sensitivity:"
)
display(
    orc_sensitivity_summary
)

print(
    "\nAlpha sensitivity:"
)
display(
    alpha_sensitivity_summary
)


# ============================================================
# 7. Plot ARI + NMI together
# ============================================================

def plot_ari_nmi_sensitivity(
    summary,
    x_col,
    x_label,
    title,
    default_value,
):

    fig, ax = plt.subplots(
        figsize=(7.5, 5.2)
    )

    ax.errorbar(
        summary[x_col],
        summary["ARI_mean"],
        yerr=summary["ARI_std"],
        marker="o",
        linewidth=2,
        capsize=4,
        label="ARI",
    )

    ax.errorbar(
        summary[x_col],
        summary["NMI_mean"],
        yerr=summary["NMI_std"],
        marker="s",
        linewidth=2,
        capsize=4,
        label="NMI",
    )

    ax.axvline(
        default_value,
        linestyle="--",
        linewidth=1.5,
        label=f"Main setting = {default_value}",
    )

    ax.set_xlabel(
        x_label
    )

    ax.set_ylabel(
        "Clustering score"
    )

    ax.set_title(
        title
    )

    ax.set_ylim(
        0,
        1,
    )

    ax.grid(
        alpha=0.3
    )

    ax.legend()

    fig.tight_layout()

    plt.show()


# LRC plot
plot_ari_nmi_sensitivity(
    summary=lrc_sensitivity_summary,
    x_col="lrc_pct",
    x_label="LRC pruning ratio (%)",
    title="SBM: ARI and NMI vs LRC pruning",
    default_value=DEFAULT_LRC_PRUNING,
)


# ORC plot
plot_ari_nmi_sensitivity(
    summary=orc_sensitivity_summary,
    x_col="orc_pct",
    x_label="ORC pruning ratio (%)",
    title="SBM: ARI and NMI vs ORC pruning",
    default_value=DEFAULT_ORC_PRUNING,
)


# Alpha plot
plot_ari_nmi_sensitivity(
    summary=alpha_sensitivity_summary,
    x_col="alpha",
    x_label=r"ORC idleness parameter $\alpha$",
    title=r"SBM: ARI and NMI vs $\alpha$",
    default_value=DEFAULT_ALPHA,
)


print(
    "\nSensitivity analysis completed."
)

In [ ]:
# ============================================================
# RANDOM PARAMETER SENSITIVITY
# LRC pruning: 0--100%
# ORC pruning: 0--100%
# alpha:       0--1
# ============================================================

import numpy as np
import pandas as pd
import time


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

N_RANDOM_CONFIGS = 50

# Graph realizations used for every random configuration
RANDOM_SENSITIVITY_SEEDS = [1, 2, 3, 4, 5]

# Reproducible random parameter sampling
PARAMETER_RANDOM_SEED = 2026

rng = np.random.default_rng(
    PARAMETER_RANDOM_SEED
)


# ============================================================
# 1. RANDOM PARAMETER CONFIGURATIONS
# ============================================================

random_configs = []

for config_id in range(N_RANDOM_CONFIGS):

    # Integer pruning percentages from 0 to 100 inclusive
    lrc_pct = int(
        rng.integers(
            0,
            101
        )
    )

    orc_pct = int(
        rng.integers(
            0,
            101
        )
    )

    # Continuous alpha between 0 and 1
    alpha = float(
        rng.uniform(
            0.0,
            1.0
        )
    )

    random_configs.append(
        {
            "config_id": config_id,
            "lrc_pct": lrc_pct,
            "orc_pct": orc_pct,
            "alpha": alpha,
        }
    )


df_random_configs = pd.DataFrame(
    random_configs
)

print("Random parameter configurations:")
display(
    df_random_configs.head(20)
)


# ============================================================
# 2. RUN RANDOM CONFIGURATIONS
# ============================================================

random_results = []


print(
    "\n"
    + "=" * 80
)

print(
    "RANDOM LRC / ORC / ALPHA SENSITIVITY"
)

print(
    "=" * 80
)


for _, config in df_random_configs.iterrows():

    config_id = int(
        config["config_id"]
    )

    lrc_pct = float(
        config["lrc_pct"]
    )

    orc_pct = float(
        config["orc_pct"]
    )

    alpha = float(
        config["alpha"]
    )


    print(
        f"\nConfig {config_id:02d} | "
        f"LRC={lrc_pct:.0f}% | "
        f"ORC={orc_pct:.0f}% | "
        f"alpha={alpha:.3f}"
    )


    for seed in RANDOM_SENSITIVITY_SEEDS:

        try:

            # -----------------------------------------------
            # Generate exactly the same synthetic graph type
            # used in the main experiment
            # -----------------------------------------------

            G, labels = generate_graph(
                seed,
                **FIXED_PARAMS,
            )


            # -----------------------------------------------
            # Run Combo pipeline
            # -----------------------------------------------

            result = run_combo(
                G=G,
                labels=labels,
                pct_lrc=lrc_pct,
                pct_orc=orc_pct,
                alpha=alpha,
                flow_steps=RICCI_FLOW_STEPS,
                eta=RICCI_FLOW_ETA,
            )


            random_results.append(
                {
                    "config_id": config_id,
                    "seed": seed,

                    "lrc_pct": lrc_pct,
                    "orc_pct": orc_pct,
                    "alpha": alpha,

                    "ARI": result["ARI"],
                    "NMI": result["NMI"],
                    "Runtime": result["Total_time"],

                    "V": result["V"],
                    "E": result["E"],

                    "removed_lrc":
                        result["removed_lrc"],

                    "removed_orc":
                        result["removed_orc"],

                    "status": "ok",
                    "error": "",
                }
            )


        except Exception as error:

            print(
                f"   Seed {seed}: FAILED -> "
                f"{type(error).__name__}"
            )

            random_results.append(
                {
                    "config_id": config_id,
                    "seed": seed,

                    "lrc_pct": lrc_pct,
                    "orc_pct": orc_pct,
                    "alpha": alpha,

                    "ARI": np.nan,
                    "NMI": np.nan,
                    "Runtime": np.nan,

                    "V": np.nan,
                    "E": np.nan,

                    "removed_lrc": np.nan,
                    "removed_orc": np.nan,

                    "status": "failed",
                    "error": repr(error),
                }
            )


df_random_sensitivity = pd.DataFrame(
    random_results
)


# ============================================================
# 3. KEEP SUCCESSFUL RUNS
# ============================================================

df_random_valid = (
    df_random_sensitivity[
        df_random_sensitivity["status"] == "ok"
    ]
    .copy()
)


print(
    "\nSuccessful runs:",
    len(df_random_valid),
)

print(
    "Failed runs:",
    (
        df_random_sensitivity["status"]
        == "failed"
    ).sum(),
)


# ============================================================
# 4. MEAN +/- STD FOR EACH RANDOM CONFIGURATION
# ============================================================

random_summary = (
    df_random_valid
    .groupby(
        [
            "config_id",
            "lrc_pct",
            "orc_pct",
            "alpha",
        ],
        as_index=False,
    )
    .agg(
        ARI_mean=("ARI", "mean"),
        ARI_std=("ARI", "std"),

        NMI_mean=("NMI", "mean"),
        NMI_std=("NMI", "std"),

        Runtime_mean=("Runtime", "mean"),
        Runtime_std=("Runtime", "std"),

        V_mean=("V", "mean"),
        E_mean=("E", "mean"),

        Removed_LRC_mean=(
            "removed_lrc",
            "mean",
        ),

        Removed_ORC_mean=(
            "removed_orc",
            "mean",
        ),

        n_runs=(
            "seed",
            "nunique",
        ),
    )
)


# ============================================================
# 5. SHOW BEST CONFIGURATIONS
# ============================================================

print(
    "\nTop configurations by ARI:"
)

display(
    random_summary
    .sort_values(
        "ARI_mean",
        ascending=False,
    )
    .head(15)
)


print(
    "\nTop configurations by NMI:"
)

display(
    random_summary
    .sort_values(
        "NMI_mean",
        ascending=False,
    )
    .head(15)
)


# ============================================================
# 6. SAVE RESULTS
# ============================================================

df_random_sensitivity.to_csv(
    OUTPUT_DIR / "sbm_random_parameter_sensitivity_raw.csv",
    index=False,
)

random_summary.to_csv(
    OUTPUT_DIR / "sbm_random_parameter_sensitivity_summary.csv",
    index=False,
)


print(
    "\nSaved:"
)

print(
    "  sbm_random_parameter_sensitivity_raw.csv"
)

print(
    "  sbm_random_parameter_sensitivity_summary.csv"
)

In [ ]:
# ============================================================
# HEATMAP TIMING WARM-UP
# ============================================================

WARMUP_TOTAL_NODES = 2500
WARMUP_RATIO = 0.15
WARMUP_TARGET_DEGREE = 25
WARMUP_ALPHA = 0.5
WARMUP_SEEDS = [0, 1]

warmup_community_size = (
    WARMUP_TOTAL_NODES
    // FIXED_PARAMS["n_communities"]
)

warmup_p_in = WARMUP_TARGET_DEGREE / (
    (warmup_community_size - 1)
    + (
        WARMUP_TOTAL_NODES
        - warmup_community_size
    )
    * WARMUP_RATIO
)

warmup_graph_params = {
    **FIXED_PARAMS,
    "community_size": warmup_community_size,
    "p_in": warmup_p_in,
    "p_out": WARMUP_RATIO * warmup_p_in,
    "bridge_blocks": max(
        FIXED_PARAMS["bridge_blocks"],
        round(
            FIXED_PARAMS["bridge_blocks"]
            * WARMUP_TOTAL_NODES
            / 360
        ),
    ),
}

for warmup_seed in WARMUP_SEEDS:
    warmup_graph, warmup_labels = generate_graph(
        seed=warmup_seed,
        **warmup_graph_params,
    )

    run_combo(
        G=warmup_graph,
        labels=warmup_labels,
        pct_lrc=20,
        pct_orc=20,
        alpha=WARMUP_ALPHA,
        flow_steps=RICCI_FLOW_STEPS,
        eta=RICCI_FLOW_ETA,
    )

print(
    "Timing warm-up complete; measured heat-map runs follow."
)

In [ ]:
# ============================================================
# JOINT LRC-ORC PRUNING HEATMAPS ACROSS SBM RATIOS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Controlled SBM parameter settings
# ------------------------------------------------------------

N_VERTICES = 2500
N_COMMUNITIES = FIXED_PARAMS["n_communities"]
COMMUNITY_SIZE = N_VERTICES // N_COMMUNITIES
TARGET_DEGREE = 25
SBM_RATIOS = [0.05, 0.10, 0.15, 0.20, 0.30]

LRC_HEAT_VALUES = [0, 10, 20, 30, 40, 50]
ORC_HEAT_VALUES = [0, 10, 20, 30, 40, 50]

HEATMAP_ALPHA = 0.5
HEATMAP_SEEDS = [1, 2, 3, 4, 5]


def probabilities_for_sbm(
    total_nodes,
    ratio,
    target_degree=25,
    n_communities=4,
):
    """Set p_in and p_out while approximately preserving mean degree."""
    community_size = total_nodes / n_communities

    p_in = target_degree / (
        (community_size - 1)
        + (total_nodes - community_size) * ratio
    )

    return p_in, ratio * p_in


SBM_CONFIGS = []

for ratio in SBM_RATIOS:
    p_in, p_out = probabilities_for_sbm(
        total_nodes=N_VERTICES,
        ratio=ratio,
        target_degree=TARGET_DEGREE,
        n_communities=N_COMMUNITIES,
    )

    SBM_CONFIGS.append(
        {
            "n_vertices": N_VERTICES,
            "community_size": COMMUNITY_SIZE,
            "ratio": ratio,
            "p_in": p_in,
            "p_out": p_out,
        }
    )


print("JOINT LRC-ORC PRUNING SENSITIVITY ACROSS SBM RATIOS")
print("vertices:", N_VERTICES)
print("community size:", COMMUNITY_SIZE)
print("target degree:", TARGET_DEGREE)
print("ratios:", SBM_RATIOS)
print("LRC values:", LRC_HEAT_VALUES)
print("ORC values:", ORC_HEAT_VALUES)
print("seeds:", HEATMAP_SEEDS)


# ============================================================
# 2. Plotting helper
# ============================================================

def plot_metric_heatmap(
    matrix,
    metric_name,
    total_nodes,
    ratio,
    runtime_matrix=None,
):
    fig, ax = plt.subplots(figsize=(8, 6))

    if metric_name == "ARI":
        vmin, vmax = -0.5, 1.0
    else:
        vmin, vmax = 0.0, 1.0

    im = ax.imshow(
        matrix.values,
        vmin=vmin,
        vmax=vmax,
        aspect="auto",
    )

    ax.set_xticks(np.arange(len(ORC_HEAT_VALUES)))
    ax.set_xticklabels([f"{value}%" for value in ORC_HEAT_VALUES])
    ax.set_yticks(np.arange(len(LRC_HEAT_VALUES)))
    ax.set_yticklabels([f"{value}%" for value in LRC_HEAT_VALUES])

    ax.set_xlabel("ORC pruning ratio (%)")
    ax.set_ylabel("LRC pruning ratio (%)")
    ax.set_title(
        rf"SBM: Joint LRC-ORC Sensitivity ({metric_name})\n"
        rf"$n={total_nodes}$, "
        rf"$p_{{out}}/p_{{in}}={ratio:.2f}$"
    )

    for i, lrc_pct in enumerate(LRC_HEAT_VALUES):
        for j, orc_pct in enumerate(ORC_HEAT_VALUES):
            metric_value = matrix.loc[lrc_pct, orc_pct]

            if runtime_matrix is not None:
                runtime_value = runtime_matrix.loc[lrc_pct, orc_pct]
                cell_text = f"{metric_value:.3f}\nT={runtime_value:.2f}s"
            else:
                cell_text = f"{metric_value:.3f}"

            ax.text(
                j,
                i,
                cell_text,
                ha="center",
                va="center",
                fontsize=10,
            )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(f"Mean {metric_name}")
    fig.tight_layout()

    return fig


# ============================================================
# 3. Run one heatmap for each ratio
# ============================================================

for config in SBM_CONFIGS:
    total_nodes = config["n_vertices"]
    ratio = config["ratio"]
    p_in = config["p_in"]
    p_out = config["p_out"]

    print(
        f"\nRunning SBM with n={total_nodes}, "
        f"p_in={p_in:.6f}, p_out={p_out:.6f}, "
        f"ratio={ratio:.2f}"
    )

    graph_params = {
        **FIXED_PARAMS,
        "community_size": config["community_size"],
        "p_in": p_in,
        "p_out": p_out,
        "bridge_blocks": max(
            FIXED_PARAMS["bridge_blocks"],
            round(
                FIXED_PARAMS["bridge_blocks"]
                * total_nodes
                / 360
            ),
        ),
    }

    heatmap_rows = []

    for lrc_pct in LRC_HEAT_VALUES:
        for orc_pct in ORC_HEAT_VALUES:
            print(
                f"  LRC={lrc_pct}% | "
                f"ORC={orc_pct}% | "
                f"ratio={ratio:.2f}"
            )

            for seed in HEATMAP_SEEDS:
                G, labels = generate_graph(
                    seed=seed,
                    **graph_params,
                )

                result = run_combo(
                    G=G,
                    labels=labels,
                    pct_lrc=lrc_pct,
                    pct_orc=orc_pct,
                    alpha=HEATMAP_ALPHA,
                    flow_steps=RICCI_FLOW_STEPS,
                    eta=RICCI_FLOW_ETA,
                )

                heatmap_rows.append(
                    {
                        "seed": seed,
                        "n_vertices": total_nodes,
                        "community_size": config["community_size"],
                        "ratio": ratio,
                        "p_in": p_in,
                        "p_out": p_out,
                        "lrc_pct": lrc_pct,
                        "orc_pct": orc_pct,
                        "alpha": HEATMAP_ALPHA,
                        "ARI": result["ARI"],
                        "NMI": result["NMI"],
                        "Runtime": result["Total_time"],
                        "V": result["V"],
                        "E": result["E"],
                        "removed_lrc": result["removed_lrc"],
                        "removed_orc": result["removed_orc"],
                    }
                )

    df_heatmap_raw = pd.DataFrame(heatmap_rows)

    heatmap_summary = (
        df_heatmap_raw
        .groupby(
            [
                "n_vertices",
                "community_size",
                "ratio",
                "p_in",
                "p_out",
                "lrc_pct",
                "orc_pct",
            ],
            as_index=False,
        )
        .agg(
            ARI_mean=("ARI", "mean"),
            ARI_std=("ARI", "std"),
            NMI_mean=("NMI", "mean"),
            NMI_std=("NMI", "std"),
            Runtime_mean=("Runtime", "mean"),
            V_mean=("V", "mean"),
            E_mean=("E", "mean"),
            Removed_LRC_mean=("removed_lrc", "mean"),
            Removed_ORC_mean=("removed_orc", "mean"),
        )
    )

    ari_matrix = (
        heatmap_summary
        .pivot(
            index="lrc_pct",
            columns="orc_pct",
            values="ARI_mean",
        )
        .reindex(
            index=LRC_HEAT_VALUES,
            columns=ORC_HEAT_VALUES,
        )
    )

    nmi_matrix = (
        heatmap_summary
        .pivot(
            index="lrc_pct",
            columns="orc_pct",
            values="NMI_mean",
        )
        .reindex(
            index=LRC_HEAT_VALUES,
            columns=ORC_HEAT_VALUES,
        )
    )

    runtime_matrix = (
        heatmap_summary
        .pivot(
            index="lrc_pct",
            columns="orc_pct",
            values="Runtime_mean",
        )
        .reindex(
            index=LRC_HEAT_VALUES,
            columns=ORC_HEAT_VALUES,
        )
    )

    fig_ari = plot_metric_heatmap(
        matrix=ari_matrix,
        metric_name="ARI",
        total_nodes=total_nodes,
        ratio=ratio,
        runtime_matrix=runtime_matrix,
    )

    fig_nmi = plot_metric_heatmap(
        matrix=nmi_matrix,
        metric_name="NMI",
        total_nodes=total_nodes,
        ratio=ratio,
        runtime_matrix=runtime_matrix,
    )

    plt.show()

    parameter_suffix = (
        f"n{total_nodes}_ratio"
        f"{str(ratio).replace('.', 'p')}"
    )

    fig_ari.savefig(
        OUTPUT_DIR / f"sbm_ari_lrc_orc_heatmap_{parameter_suffix}.png",
        dpi=300,
        bbox_inches="tight",
    )

    fig_nmi.savefig(
        OUTPUT_DIR / f"sbm_nmi_lrc_orc_heatmap_{parameter_suffix}.png",
        dpi=300,
        bbox_inches="tight",
    )

    heatmap_summary.to_csv(
        OUTPUT_DIR / f"sbm_lrc_orc_heatmap_{parameter_suffix}_summary.csv",
        index=False,
    )

    df_heatmap_raw.to_csv(
        OUTPUT_DIR / f"sbm_lrc_orc_heatmap_{parameter_suffix}_raw.csv",
        index=False,
    )

    plt.close(fig_ari)
    plt.close(fig_nmi)

    print(
        "  Saved ARI and NMI heatmaps plus CSV results."
    )

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

np.random.seed(42)

# toy graph: two communities connected by bridges
G = nx.Graph()
left = range(0, 12)
right = range(12, 24)

G.add_nodes_from(left)
G.add_nodes_from(right)

for group in [left, right]:
    for i in group:
        for j in group:
            if i < j and np.random.rand() < 0.45:
                G.add_edge(i, j, weight=1.0)

bridges = [(3, 15), (5, 18), (8, 20)]
G.add_edges_from(bridges, weight=1.0)

pos = nx.spring_layout(G, seed=4)

# fake curvature values for illustration
for u, v in G.edges():
    if (u, v) in bridges or (v, u) in bridges:
        G[u][v]["curvature"] = -0.55
    else:
        G[u][v]["curvature"] = 0.25 + 0.15 * np.random.rand()

eta = 0.8
steps = [0, 1, 3, 6]

graphs = []

for t in range(max(steps) + 1):
    H = G.copy()
    for u, v in H.edges():
        kappa = H[u][v]["curvature"]
        H[u][v]["weight"] = G[u][v]["weight"] * ((1 - eta * kappa) ** t)
    graphs.append(H)

fig, axes = plt.subplots(1, len(steps), figsize=(18, 4))

for ax, t in zip(axes, steps):
    H = graphs[t]
    widths = [0.8 + 2.8 * H[u][v]["weight"] for u, v in H.edges()]
    
    edge_colors = [
        "crimson" if H[u][v]["curvature"] < 0 else "gray"
        for u, v in H.edges()
    ]

    node_colors = ["#377eb8" if n in left else "#4daf4a" for n in H.nodes()]

    nx.draw_networkx_edges(
        H, pos, ax=ax,
        width=widths,
        edge_color=edge_colors,
        alpha=0.65
    )
    nx.draw_networkx_nodes(
        H, pos, ax=ax,
        node_color=node_colors,
        node_size=180,
        edgecolors="black",
        linewidths=0.5
    )

    ax.set_title(f"Ricci flow step t={t}")
    ax.axis("off")

plt.suptitle("Discrete Ricci Flow Dynamics on a Network", fontsize=16)
plt.tight_layout()
plt.show()

fig.savefig(OUTPUT_DIR / "fig4_discrete_ricci_flow_dynamics.png", dpi=400, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "fig4_discrete_ricci_flow_dynamics.pdf", bbox_inches="tight")

In [ ]:
configs_for_paper = [
    (0,  10, 0.50),
    (10, 30, 0.25),
    (20, 25, 0.75),
    (25, 30, 0.75),
    (30, 20, 0.50),
]

paper_rows = []

for lrc_pct, orc_pct, alpha in configs_for_paper:

    rows = evaluate_combo_configuration(
        lrc_pct=lrc_pct,
        orc_pct=orc_pct,
        alpha=alpha,
        seeds=SENSITIVITY_SEEDS,
    )

    df_temp = pd.DataFrame(rows)

    paper_rows.append({
        "p_LRC": lrc_pct,
        "p_ORC": orc_pct,
        "alpha": alpha,
        "ARI": df_temp["ARI"].mean(),
        "NMI": df_temp["NMI"].mean(),
        "Runtime": df_temp["Runtime"].mean(),
    })

paper_table = pd.DataFrame(paper_rows)

display(paper_table.round(3))